# Reproduce the manuscript insulation figures

This notebook generates four paired left/right heatmaps with mean profiles and no profile legends:

| Output | Manuscript figure | Tissue / selection |
| --- | --- | --- |
| plot1 | Fig. 1h | Embryo, all peaks / overlap subsets |
| plot2 | Fig. S3b | Larval brain, all peaks / overlap subsets |
| plot3 | Fig. 3b | Embryo, motif-selected peaks |
| plot4 | Fig. S5a,b | Larval brain, motif-selected peaks |

**400 bp contact-map resolution; 1,200 bp insulation window; ±10 kb around peak midpoints; 400 bp profile bins.** These match the saved `window1200bp` matrices. No 600 bp settings are used. This notebook does not generate Fig. 4 locus views; those are in `Figure4_pyGenomeTracks.ipynb`.

## Two modes

- Default: `REBUILD_FROM_MAPS = False` uses the previously calculated deepTools matrices in `output/insulation/matrices`. It does not recompute insulation.
- Reproduction: set `REBUILD_FROM_MAPS = True` and point `COOL_FILES` to the published embryo and larval-brain maps. Each map must contain a 400 bp resolution with balancing weights named `weight`. The supplied BED files must be present in `BASE`, with chromosome names and coordinates matching the map assembly. The workflow computes scores, exports bigWigs and runs deepTools. New results go into a separate directory; existing results are not overwritten.

Requirements used for the original analysis: Python 3.10, cooler 0.9.3, cooltools 0.7.1, deepTools 3.5.5, pyBigWig 0.3.22, numpy 1.26.4, pandas, matplotlib and Pillow. Public accession identifiers and map preprocessing details must accompany the published data; local filenames alone do not identify a public dataset. This notebook starts from processed balanced maps, not sequencing reads.

## Insulation calculation

For each chromosome, cooltools averages balanced contacts within a diamond spanning each position, using a three-bin (1.2 kb) window. Pixels involving missing weights and contacts separated by fewer than two bins are excluded (`ignore_diags=2`). Scores are divided by the chromosome-wide median and log2-transformed; zero pre-log scores and non-finite results become missing. Lower scores indicate stronger insulation. The plotted quantity is `log2_insulation_score_1200`, not boundary strength. Existing balancing weights are used without rebalancing.

Finite scores are exported to bigWig. `computeMatrix reference-point --referencePoint center` summarizes ±10 kb around the supplied BED midpoints. Missing values are not replaced with zero; valid log2 zeros are retained. Motif-selected peak intervals are not recentered on individual motif occurrences. Tissue-matched tracks are used. BED order is preserved without clustering. `plotHeatmap` displays mean profiles and the installed deepTools default colormap (use the original version for consistent appearance).

Figure color and profile limits are fixed to the original 1.2 kb plotting command, rather than read from the later overwritten `run_parameters.json`. These are display limits only, not filters on the scores. The source maps' original 400 bp weight metadata reported `converged=False`; review balancing QC when preparing published maps.

**Prepared without execution.** Run the cells in order after checking paths.


## Portable paths

Launch Jupyter from the repository root. Input files are expected under `data/` by default. Set `SRY_DATA_DIR`, `SRY_MCOOL_DIR` and (for Fig. 4) `SRY_GTF`, or edit the configuration cell to use other locations. Inputs are not bundled with this notebook. See `README.md` for the directory layout. Source data filenames and analysis parameters are retained. Saved outputs were cleared for publication; no analysis was rerun.


In [ ]:
from pathlib import Path
import os, sys, json, subprocess, shutil, hashlib, importlib.metadata
# Run from the repository root; override paths here or with environment variables.
PROJECT_DIR = Path.cwd()
BASE = Path(os.environ.get('SRY_DATA_DIR', PROJECT_DIR / 'data')).expanduser().resolve()
MCOOL_DIR = Path(os.environ.get('SRY_MCOOL_DIR', BASE / 'mcool')).expanduser().resolve()
REBUILD_FROM_MAPS = False
CACHED = BASE / 'output' / 'insulation'
OUT = BASE / 'output' / 'insulation_reproduction_1200bp'
FIG = BASE / 'figures' / 'insulation_manuscript_1200bp'
for folder in (OUT, FIG, OUT/'tracks', OUT/'beds', OUT/'matrices', OUT/'logs'):
    folder.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(OUT / '.matplotlib')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cooler, cooltools, pyBigWig
from cooltools import insulation
from IPython.display import display, Image
RESOLUTION = 400
WINDOWS = [1200]
FLANK = 10000
BIN_SIZE = RESOLUTION
THREADS = 4
FORCE_RECOMPUTE = False
COOL_FILES = {
    'embryo': MCOOL_DIR / 'AllWT_MJP_balanced.mcool',
    'LB22': MCOOL_DIR / 'larval_brain_merge.mcool',
}
TOOLS = {}
for name in (('computeMatrix', 'plotHeatmap') if REBUILD_FROM_MAPS else ('plotHeatmap',)):
    candidate = Path(sys.executable).parent / name
    TOOLS[name] = str(candidate) if candidate.exists() else shutil.which(name)
    if not TOOLS[name]:
        raise RuntimeError(f'Missing {name}: install deepTools in this kernel environment.')
assert FLANK % BIN_SIZE == 0
versions = {p: importlib.metadata.version(p) for p in ('cooler','cooltools','deepTools','pyBigWig','numpy')}
print(versions)

DATA = OUT if REBUILD_FROM_MAPS else CACHED
WINDOW = WINDOWS[0]


### Validate peak groups and load maps

Runs only when `REBUILD_FROM_MAPS = True`.

In [ ]:
if REBUILD_FROM_MAPS:
    def galaxy(number):
        matches = list(BASE.glob(f'Galaxy{number}-*.bed'))
        if len(matches) != 1:
            raise ValueError(f'Expected one BED for Galaxy{number}, found {matches}')
        return matches[0]
    S = 'Sry-delta'
    B = 'BEAF32'
    all_labels = [f'{S} all peaks', f'{B} all peaks']
    subset_labels = [f'{S}(+){B}', f'{B}(-){S}', f'{S}(-){B}']
    motif_labels = [f'{S} peaks only with {S} MEME motifs', f'{B} peaks only with {B} MEME motifs']
    shared_labels = [f'{S}(+){B} with {S} and BEAF MEME motifs', f'{S}(+){B} only with {S} MEME motifs', f'{S}(+){B} only with {B} MEME motifs']
    PANELS = {
     'plot1_left': ('embryo', [galaxy(4554), galaxy(4557)], all_labels),
     'plot1_right': ('embryo', [galaxy(4581), galaxy(4583), galaxy(4582)], subset_labels),
     'plot2_left': ('LB22', [galaxy(4528), galaxy(4571)], all_labels),
     'plot2_right': ('LB22', [galaxy(4593), galaxy(4596), galaxy(4594)], subset_labels),
     'plot3_left': ('embryo', [BASE/'Sry embryo peaks with Srymotifs coordinates.bed', BASE/'BEAF embryo peaks with BEAFmotifs coordinates.bed'], motif_labels),
     'plot3_right': ('embryo', [galaxy(13997), galaxy(13998), galaxy(13999)], shared_labels),
     'plot4_left': ('LB22', [BASE/'Sry LB22 peaks with Srymotif coordinates.bed', BASE/'BEAF LB22 peaks with BEAFmotifs coordinates.bed'], motif_labels),
     'plot4_right': ('LB22', [galaxy(14009), galaxy(14010), galaxy(14011)], shared_labels),
    }
    clrs = {t: cooler.Cooler(f'{p}::resolutions/{RESOLUTION}') for t,p in COOL_FILES.items()}
    manifest = []
    BEDS = {}
    for panel, (tissue, files, labels) in PANELS.items():
        BEDS[panel] = []
        for i, (file, label) in enumerate(zip(files, labels)):
            bed = pd.read_csv(file, sep='\t', header=None, usecols=[0,1,2], names=['chrom','start','end'])
            sizes = bed.chrom.map(clrs[tissue].chromsizes)
            valid = sizes.notna() & (bed.start >= 0) & (bed.end > bed.start) & (bed.end <= sizes)
            valid &= (bed.start % 1 == 0) & (bed.end % 1 == 0)
            if not valid.all() or bed.empty:
                raise ValueError(f'Invalid/empty intervals in {file}: {bed.loc[~valid].head()}')
            # BED3 intentionally prevents accidental strand reversal; preserve duplicates and order.
            target = OUT/'beds'/f'{panel}_{i+1}.bed'
            bed.to_csv(target, sep='\t', header=False, index=False)
            BEDS[panel].append(target)
            manifest.append(dict(panel=panel, tissue=tissue, order=i+1, source=file.name,
                                 label=label, intervals=len(bed), duplicate_rows=int(bed.duplicated().sum())))
    manifest = pd.DataFrame(manifest)
    manifest.to_csv(OUT/'interval_manifest.tsv', sep='\t', index=False)
    display(manifest)
else:
    print('Using previously calculated results; rebuild step skipped.')


### Calculate / load insulation scores and export bigWigs

Runs only when `REBUILD_FROM_MAPS = True`.

In [ ]:
if REBUILD_FROM_MAPS:
    tables = {}
    tracks = {}
    for tissue, clr in clrs.items():
        source = COOL_FILES[tissue]
        signature = dict(source=str(source), size=source.stat().st_size, mtime_ns=source.stat().st_mtime_ns,
                         resolution=RESOLUTION, windows=WINDOWS, ignore_diags=2, cooltools=versions['cooltools'])
        cache = OUT/f'{tissue}_insulation.tsv.gz'
        meta = OUT/f'{tissue}_insulation.json'
        if not FORCE_RECOMPUTE and cache.exists() and meta.exists() and json.loads(meta.read_text()) == signature:
            table = pd.read_csv(cache, sep='\t')
            print(f'{tissue}: loaded cached insulation', flush=True)
        else:
            print(f'{tissue}: computing insulation', flush=True)
            table = insulation(clr, WINDOWS, ignore_diags=2, verbose=True)
            table.to_csv(cache, sep='\t', index=False)
            meta.write_text(json.dumps(signature, indent=2))
        tables[tissue] = table
        tracks[tissue] = []
        for window in WINDOWS:
            col = f'log2_insulation_score_{window}'
            path = OUT/'tracks'/f'{tissue}_log2_insulation_{window}.bw'
            with pyBigWig.open(str(path), 'w') as bw:
                bw.addHeader([(str(chrom), int(size)) for chrom,size in clr.chromsizes.items()])
                for chrom in clr.chromnames:
                    rows = table.loc[(table.chrom == chrom) & np.isfinite(table[col])].sort_values('start')
                    if len(rows):
                        bw.addEntries(rows.chrom.astype(str).tolist(), rows.start.astype(int).tolist(),
                                      ends=rows.end.astype(int).tolist(), values=rows[col].astype(float).tolist())
            tracks[tissue].append(path)
        print(f'{tissue}: exported {len(WINDOWS)} insulation bigWigs', flush=True)
    # Compatibility with the original notebook variable names.
    insulation_table = tables['embryo']
    insulation_table2 = tables['LB22']
else:
    print('Using previously calculated results; rebuild step skipped.')


### Extract matrices and export profile coverage

Runs only when `REBUILD_FROM_MAPS = True`.

In [ ]:
if REBUILD_FROM_MAPS:
    def run_tool(name, args, log_name):
        command = [TOOLS[name], *map(str, args)]
        with (OUT/'logs'/'commands.jsonl').open('a') as log:
            log.write(json.dumps(command) + '\n')
        result = subprocess.run(command, capture_output=True, text=True)
        (OUT/'logs'/log_name).write_text(result.stdout + result.stderr)
        if result.returncode:
            raise RuntimeError(f'{name} failed: {result.stderr[-4000:]}')
    
    # Symmetric, shared heatmap limits across both tissues and all windows.
    values = np.concatenate([t[f'log2_insulation_score_{w}'].to_numpy() for t in tables.values() for w in WINDOWS])
    COLOR_LIMIT = float(np.quantile(np.abs(values[np.isfinite(values)]), 0.98))
    if COLOR_LIMIT == 0: COLOR_LIMIT = 1.0
    sample_labels = [f'{w/1000:g} kb window' for w in WINDOWS]
    matrices = {}
    coverage = []
    for panel, (tissue, files, labels) in PANELS.items():
        matrix = OUT/'matrices'/f'{panel}.mat.gz'
        run_tool('computeMatrix', ['reference-point', '--referencePoint', 'center',
            '-S', *tracks[tissue], '-R', *BEDS[panel], '-b', FLANK, '-a', FLANK,
            '--binSize', BIN_SIZE, '--sortRegions', 'keep', '--numberOfProcessors', THREADS,
            '--samplesLabel', *sample_labels, '-o', matrix,
            '--outFileSortedRegions', OUT/'matrices'/f'{panel}.regions.bed'], f'{panel}_computeMatrix.log')
        import gzip
        with gzip.open(matrix, 'rt') as stream:
            header = json.loads(stream.readline()[1:])
            data = np.loadtxt(stream, delimiter='\t', usecols=range(6, 6 + header['sample_boundaries'][-1]), ndmin=2)
        expected = manifest.loc[manifest.panel == panel, 'intervals'].tolist()
        assert np.diff(header['group_boundaries']).tolist() == expected, (panel, 'Region count mismatch')
        matrices[panel] = matrix
        for group, label in enumerate(labels):
            for sample, window in enumerate(WINDOWS):
                block = data[header['group_boundaries'][group]:header['group_boundaries'][group+1],
                             header['sample_boundaries'][sample]:header['sample_boundaries'][sample+1]]
                for j, count in enumerate(np.isfinite(block).sum(axis=0)):
                    coverage.append(dict(panel=panel, label=label, window=window,
                        offset_bp=-FLANK+(j+0.5)*BIN_SIZE, n_valid=int(count), n_total=len(block)))
        print(f'{panel}: matrix verified ({len(data)} intervals)', flush=True)
    pd.DataFrame(coverage).to_csv(OUT/'profile_coverage.tsv', sep='\t', index=False)
    # Shared profile y limits across all panels, based on the actual group means.
    means = []
    for panel,matrix in matrices.items():
        with gzip.open(matrix, 'rt') as stream:
            h=json.loads(stream.readline()[1:])
            a=np.loadtxt(stream, delimiter='\t', usecols=range(6,6+h['sample_boundaries'][-1]), ndmin=2)
        for lo,hi in zip(h['group_boundaries'][:-1],h['group_boundaries'][1:]):
            means.extend(np.nanmean(a[lo:hi],axis=0))
    finite_means=np.asarray(means)[np.isfinite(means)]
    padding=max(0.05, float(np.ptp(finite_means))*0.08)
    YMIN,YMAX=float(finite_means.min()-padding),float(finite_means.max()+padding)
    (OUT/'run_parameters.json').write_text(json.dumps(dict(resolution=RESOLUTION, windows=WINDOWS,
        flank=FLANK, bin_size=BIN_SIZE, color_limit=COLOR_LIMIT, profile_ylim=[YMIN,YMAX],
        versions=versions, sources={k:str(v) for k,v in COOL_FILES.items()}), indent=2))
else:
    print('Using previously calculated results; rebuild step skipped.')


### Verify figure inputs and restore the original display settings

In [ ]:
import csv, gzip, textwrap, tempfile
from PIL import Image as PILImage
PLOT_HEATMAP = TOOLS['plotHeatmap']
# Fixed original display limits, recovered from the 1.2 kb plotting log.
COLOR_LIMIT = 1.3501449903752447
YMIN, YMAX = -0.4890547529510921, 0.7865106355481832
with (DATA / 'interval_manifest.tsv').open() as stream:
    rows = list(csv.DictReader(stream, delimiter='\t'))
PANELS = {}
for number in range(1, 5):
    for side in ('left', 'right'):
        panel = f'plot{number}_{side}'
        groups = sorted((r for r in rows if r['panel'] == panel), key=lambda r: int(r['order']))
        matrix = DATA / 'matrices' / (f'{panel}.mat.gz' if REBUILD_FROM_MAPS else f'{panel}_window{WINDOW}bp.mat.gz')
        if not matrix.exists():
            raise FileNotFoundError(f'Missing cached matrix: {matrix}. Set REBUILD_FROM_MAPS = True to regenerate it.')
        with gzip.open(matrix, 'rt') as stream:
            header = json.loads(stream.readline()[1:])
        assert header['sample_labels'] == [f'{WINDOW/1000:g} kb window']
        assert header['bin size'] == [RESOLUTION]
        assert header['upstream'] == header['downstream'] == [FLANK]
        assert header['ref point'] == ['center']
        assert not header['missing data as zero'] and not header['skip zeros']
        counts = [b-a for a,b in zip(header['group_boundaries'][:-1], header['group_boundaries'][1:])]
        assert counts == [int(r['intervals']) for r in groups], panel
        PANELS[panel] = (matrix, groups[0]['tissue'], [r['label'] for r in groups])
print('Verified all eight input panels for the four 1.2 kb figures.')


In [ ]:
def render_panel(panel, output, show_legend):
    matrix, tissue, labels = PANELS[panel]
    wrapped = ['\n'.join(textwrap.wrap(label, 32)) for label in labels]
    command = [PLOT_HEATMAP, '-m', str(matrix), '-o', str(output),
        '--regionsLabel', *wrapped, '--samplesLabel', f'{WINDOW/1000:g} kb window',
        '--refPointLabel', 'Peak center', '--plotTitle', f'{panel} | {tissue} | {WINDOW/1000:g} kb window',
        '--dpi', '150', '--yMin', str(YMIN), '--yMax', str(YMAX),
        '--yAxisLabel', 'Mean log2 insulation', '--sortRegions', 'keep',
        '--missingDataColor', '#bdbdbd', '--zMin', str(-COLOR_LIMIT), '--zMax', str(COLOR_LIMIT),
        '--heatmapHeight', '18', '--heatmapWidth', '9', '--averageTypeSummaryPlot', 'mean',
        '--plotType', 'lines', '--xAxisLabel', 'Distance from peak center (bp)',
        '--whatToShow', 'plot, heatmap and colorbar',
        '--legendLocation', 'best' if show_legend else 'none']
    # Omit --colorMap to retain the deepTools default.
    result = subprocess.run(command, capture_output=True, text=True)
    if result.returncode:
        raise RuntimeError(result.stderr)

outputs = []
with tempfile.TemporaryDirectory(prefix='sry-insulation-') as tmp:
    for number in range(1, 5):
        for show_legend in (False,):
            images = []
            for side in ('left', 'right'):
                path = Path(tmp) / f'{side}.png'
                render_panel(f'plot{number}_{side}', path, show_legend)
                with PILImage.open(path) as image:
                    images.append(image.convert('RGB'))
            left, right = images
            canvas = PILImage.new('RGB', (left.width + right.width + 30, max(left.height, right.height)), 'white')
            canvas.paste(left, (0, 0))
            canvas.paste(right, (left.width + 30, 0))
            suffix = '' if show_legend else '_no_profile_labels'
            output = FIG / f'plot{number}_window{WINDOW}bp_heatmap_left_right{suffix}.png'
            canvas.save(output)
            outputs.append(output)
        print(f'Plot {number}: saved without profile legends.', flush=True)
assert len(outputs) == 4 and all(path.exists() for path in outputs)
print(f'Saved exactly four PNG figures to {FIG}')

# Record the actual matrix inputs and plotting settings alongside the new figures.
provenance = dict(mode='rebuild' if REBUILD_FROM_MAPS else 'cached',
    resolution=RESOLUTION, insulation_window=WINDOW, flank=FLANK,
    color_limit=COLOR_LIMIT, profile_ylim=[YMIN,YMAX], versions=versions,
    matrices={k:dict(path=str(v[0]), sha256=hashlib.sha256(v[0].read_bytes()).hexdigest())
              for k,v in PANELS.items()}, outputs=[str(p) for p in outputs])
(FIG/'figure_provenance.json').write_text(json.dumps(provenance, indent=2))


In [ ]:
for output in outputs:
    print(output.name)
    display(Image(filename=str(output), width=1000))
